![Universidad Espíritu Santo](https://raw.githubusercontent.com/andresrubiop/miar0525-estudiantes/main/utils/logo-uees-color.png)

<div style="background:#821436;color:#FFFFFF;padding:14px 18px;border-radius:10px;margin:6px 0 12px 0"><div style="font-size:12px;letter-spacing:.08em;text-transform:uppercase;opacity:.9">Aprendizaje Automático · MIAR0525 · Semana 3 · Notebook del estudiante · no calificable</div><div style="font-size:22px;font-weight:700;margin-top:4px">E3.2 · DBSCAN y clustering por densidad</div><div style="font-size:12px;opacity:.9;margin-top:4px">Postgrado · Maestría en Inteligencia Artificial · UEES</div></div>

| | |
|---|---|
| **Objetivo** | Programar DBSCAN, elegir eps con el gráfico de k-distancia y agrupar coordenadas geográficas con la distancia haversine. |
| **Resultado de aprendizaje** | RDA2 · competencias CG-G2 y CE-G2 (según el sílabo) |
| **Duración** | ≈ 3.5 h |
| **Teoría** | Manual M3 §4–5 · Animaciones A3.3 y A3.4 · Video V3.1 |
| **Datos** | Sintéticos (`make_moons`) · **California Housing** (`fetch_california_housing`: coordenadas de 20 640 distritos censales) |

**Niveles:** 1 · DBSCAN desde cero → 2 · `DBSCAN`, k-distancia y comparación con K-means → 3 · clustering geográfico con haversine → 4 · reto: sensibilidad y estabilidad.

## 0 · Configuración

In [ ]:
import warnings
from collections import deque

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from cycler import cycler

warnings.filterwarnings("ignore", category=FutureWarning)
SEED = 2026
UEES = {"vino": "#821436", "azul": "#1F6F8B", "ocre": "#C28E0E", "verde": "#3A7D44", "gris": "#77787B"}
plt.rcParams.update({
    "axes.prop_cycle": cycler(color=list(UEES.values())), "axes.titlecolor": UEES["vino"],
    "axes.titleweight": "bold", "axes.edgecolor": UEES["gris"], "axes.grid": True, "grid.color": "#EEE8EA",
    "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110, "legend.frameon": False,
})
PALETA = [UEES["vino"], UEES["azul"], UEES["ocre"], UEES["verde"], "#5FB3CE", "#E07A9B", "#6B102C", "#9BBF85", "#1C1A1B"]
print(f"scikit-learn {sklearn.__version__} · semilla {SEED}")

## Nivel 1 · Desde cero (prelaboratorio)

DBSCAN define los clusters como **regiones densas**. Con dos parámetros, $\varepsilon$ (radio) y `minPts`:
- un punto es **núcleo** si su ε-vecindad (incluido él mismo) tiene al menos `minPts` puntos;
- un punto es **borde** si no es núcleo pero está en la ε-vecindad de un núcleo;
- el resto es **ruido** (etiqueta −1).

Un cluster se construye por **expansión en anchura**: se parte de un núcleo, se agregan sus vecinos y se sigue expandiendo solo desde los que también son núcleo.

In [ ]:
from sklearn.datasets import make_moons

Xm, ym = make_moons(n_samples=400, noise=0.08, random_state=SEED)
rng = np.random.default_rng(SEED)
Xm = np.vstack([Xm, rng.uniform([-1.4, -0.9], [2.4, 1.4], size=(25, 2))])      # 25 puntos de ruido
ym = np.concatenate([ym, -np.ones(25, int)])


def dbscan_propio(X, eps, min_pts):
    """Devuelve (etiquetas con −1 para ruido, máscara de núcleos)."""
    n = len(X)
    d = np.sqrt(((X[:, None, :] - X[None, :, :]) ** 2).sum(axis=2))
    vecinos = [np.flatnonzero(d[i] <= eps) for i in range(n)]
    # TODO: marca los núcleos y expande cada cluster en anchura (BFS) desde un núcleo no visitado.
    nucleo = ...
    etiquetas = -np.ones(n, int)
    cluster = 0
    for i in range(n):
        ...
    return etiquetas, nucleo


etq_p, nuc_p = dbscan_propio(Xm, eps=0.15, min_pts=6)
tipo = np.where(nuc_p, "núcleo", np.where(etq_p >= 0, "borde", "ruido"))
print(f"clusters: {etq_p.max() + 1} · núcleos {np.sum(tipo == 'núcleo')} · bordes {np.sum(tipo == 'borde')} · ruido {np.sum(tipo == 'ruido')}")

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.2))
for t, (mk, s) in {"núcleo": ("o", 22), "borde": ("o", 9), "ruido": ("x", 26)}.items():
    m = tipo == t
    colores = ["#1C1A1B" if e < 0 else PALETA[e % len(PALETA)] for e in etq_p[m]]
    ax.scatter(Xm[m, 0], Xm[m, 1], c=colores, marker=mk, s=s, label=t, alpha=0.9 if t != "borde" else 0.6)
ax.set(title="DBSCAN desde cero (eps = 0.15, minPts = 6)", xlabel="x₁", ylabel="x₂")
ax.legend()
plt.show()

## Nivel 2 · Con scikit-learn

In [ ]:
from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics import adjusted_rand_score
from sklearn.neighbors import NearestNeighbors

db = DBSCAN(eps=0.15, min_samples=6).fit(Xm)
print(f"scikit-learn: {db.labels_.max() + 1} clusters · ruido {np.sum(db.labels_ == -1)} · ARI con tu versión {adjusted_rand_score(db.labels_, etq_p):.3f}")
assert np.array_equal(db.labels_ == -1, etq_p == -1), "El ruido no coincide con scikit-learn."
assert np.array_equal(np.sort(db.core_sample_indices_), np.flatnonzero(nuc_p)), "Los núcleos no coinciden."
print("✓ Tu DBSCAN coincide con scikit-learn")

### 2.1 Elegir eps: el gráfico de k-distancia
Para cada punto calculamos la distancia a su k-ésimo vecino más cercano (k = `minPts`) y las ordenamos. El **codo** de la curva separa los puntos en zonas densas (distancias pequeñas) del ruido (distancias grandes): es un buen candidato para eps.

In [ ]:
k = 6
dist, _ = NearestNeighbors(n_neighbors=k).fit(Xm).kneighbors(Xm)      # incluye al propio punto, como minPts
kdist = np.sort(dist[:, -1])
fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.plot(kdist, color=UEES["vino"])
ax.axhline(0.15, ls="--", color=UEES["azul"], label="eps = 0.15")
ax.set(title=f"Gráfico de {k}-distancia (ordenado)", xlabel="puntos ordenados", ylabel=f"distancia al {k}.º vecino")
ax.legend()
plt.show()
print(f"percentil 90 de la k-distancia: {np.quantile(kdist, 0.90):.3f} · percentil 95: {np.quantile(kdist, 0.95):.3f}")

In [ ]:
km = KMeans(n_clusters=2, n_init=10, random_state=SEED).fit(Xm)
mascara = ym >= 0
comparacion = pd.Series({"K-means (k = 2)": adjusted_rand_score(ym[mascara], km.labels_[mascara]),
                         "DBSCAN (eps = 0.15)": adjusted_rand_score(ym[mascara], db.labels_[mascara])}, name="ARI con las lunas reales").round(3)
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8))
axes[0].scatter(Xm[:, 0], Xm[:, 1], c=[PALETA[e] for e in km.labels_], s=10)
axes[0].set(title=f"K-means · ARI {comparacion.iloc[0]:.2f}")
axes[1].scatter(Xm[:, 0], Xm[:, 1], c=["#1C1A1B" if e < 0 else PALETA[e] for e in db.labels_], s=10)
axes[1].set(title=f"DBSCAN · ARI {comparacion.iloc[1]:.2f} · ruido en negro")
plt.tight_layout()
plt.show()
comparacion

**Qué observar.** K-means supone clusters convexos y parte las lunas por la mitad; DBSCAN sigue la forma y además aparta el ruido. A cambio, DBSCAN depende mucho de eps y le cuesta cuando los clusters tienen **densidades distintas**.

## Nivel 3 · Datos reales: dónde se concentran los distritos de California

Agrupamos los distritos censales por su **ubicación**. Con coordenadas geográficas la distancia euclidiana en grados es incorrecta: usamos la distancia **haversine** (sobre la esfera), que en scikit-learn requiere latitud y longitud **en radianes** y un eps en radianes: $\varepsilon = \text{km} / 6371$.

In [ ]:
from sklearn.datasets import fetch_california_housing

cal = fetch_california_housing(as_frame=True).frame
coords = np.radians(cal[["Latitude", "Longitude"]].to_numpy())
R_TIERRA = 6371.0


def dbscan_geo(eps_km, min_samples):
    return DBSCAN(eps=eps_km / R_TIERRA, min_samples=min_samples, metric="haversine", algorithm="ball_tree").fit(coords).labels_


etq_geo = dbscan_geo(eps_km=5, min_samples=60)
cal["zona"] = etq_geo
resumen_geo = cal[cal.zona >= 0].groupby("zona").agg(distritos=("MedHouseVal", "size"), lat=("Latitude", "mean"), lon=("Longitude", "mean"),
                                                     valor_mediano=("MedHouseVal", "median"), ingreso_mediano=("MedInc", "median"))
resumen_geo = resumen_geo.sort_values("distritos", ascending=False).round(2)
print(f"{resumen_geo.shape[0]} zonas densas · {np.mean(etq_geo == -1):.1%} de los distritos quedan como ruido (zonas poco pobladas)")
resumen_geo.head(8)

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 6.4))
ruido = etq_geo == -1
ax.scatter(cal.Longitude[ruido], cal.Latitude[ruido], s=2, c="#CFC8CB", label="ruido")
orden = {z: i for i, z in enumerate(resumen_geo.index)}
ax.scatter(cal.Longitude[~ruido], cal.Latitude[~ruido], s=3, c=[PALETA[orden[z] % len(PALETA)] for z in etq_geo[~ruido]])
for i, (z, fila) in enumerate(resumen_geo.head(6).iterrows()):
    ax.annotate(f"zona {z} · {int(fila.distritos)} distritos", (fila.lon, fila.lat), fontsize=7, textcoords="offset points",
                xytext=(10, 6) if i % 2 == 0 else (10, -14), arrowprops={"arrowstyle": "-", "color": "#77787B", "lw": 0.6})
ax.set(title="DBSCAN con haversine (eps = 5 km, minPts = 60)", xlabel="longitud", ylabel="latitud")
plt.show()

**Qué observar.** Las zonas densas coinciden con las áreas metropolitanas (Los Ángeles, la bahía de San Francisco, San Diego, Sacramento…) sin haberle dicho al algoritmo cuántas hay. Los distritos rurales quedan como "ruido": no son errores, sino zonas poco densas. Compara los valores medianos: la ubicación explica buena parte del precio, lo que E2.1 no pudo capturar con un modelo lineal.

## Nivel 4 · Reto: ¿qué tan sensible es el resultado?

In [ ]:
filas = []
for eps_km in (3, 5, 10, 20):
    for ms in (30, 60, 120):
        e = dbscan_geo(eps_km, ms)
        filas.append({"eps (km)": eps_km, "minPts": ms, "zonas": int(e.max() + 1), "% ruido": round(np.mean(e == -1) * 100, 1),
                      "% en la zona mayor": round(np.max(np.bincount(e[e >= 0])) / len(e) * 100, 1) if np.any(e >= 0) else 0})
tabla_sens = pd.DataFrame(filas)
tabla_sens.pivot(index="eps (km)", columns="minPts", values="zonas")

In [ ]:
tabla_sens.pivot(index="eps (km)", columns="minPts", values="% ruido")

In [ ]:
def estabilidad_db(eps_km, ms, repeticiones=5):
    """ARI entre las etiquetas de DBSCAN en submuestras del 80 %, comparado sobre los puntos comunes."""
    r = np.random.default_rng(SEED)
    n = len(coords)
    soluciones = []
    for _ in range(repeticiones):
        idx = np.sort(r.choice(n, int(0.8 * n), replace=False))
        etiquetas = np.full(n, -2)
        etiquetas[idx] = DBSCAN(eps=eps_km / R_TIERRA, min_samples=int(ms * 0.8), metric="haversine", algorithm="ball_tree").fit(coords[idx]).labels_
        soluciones.append(etiquetas)
    aris = []
    for i in range(repeticiones):
        for j in range(i + 1, repeticiones):
            comun = (soluciones[i] != -2) & (soluciones[j] != -2)
            aris.append(adjusted_rand_score(soluciones[i][comun], soluciones[j][comun]))
    return float(np.mean(aris))


estab = {f"eps {e} km · minPts {m}": round(estabilidad_db(e, m), 3) for e, m in ((5, 60), (10, 60), (20, 30))}
estab

**Qué observar.** Con eps pequeño y minPts grande aparecen muchas zonas pequeñas y la mayoría de los distritos queda como ruido; con eps = 20 km el ruido casi desaparece y una sola zona (el sur de California, de San Diego a Santa Bárbara) reúne más de la mitad de los distritos. No existe el eps "correcto" en abstracto: depende de la pregunta (¿barrios, ciudades o regiones?). La estabilidad frente a submuestras es alta en las configuraciones razonables, lo que da confianza en las zonas encontradas.

## Autoverificación

In [ ]:
assert np.array_equal(db.labels_ == -1, etq_p == -1)
assert comparacion["DBSCAN (eps = 0.15)"] > comparacion["K-means (k = 2)"] + 0.3, "DBSCAN debería seguir la forma de las lunas mucho mejor que K-means."
assert resumen_geo.shape[0] >= 5, "Con eps = 5 km deberían aparecer varias zonas metropolitanas."
assert tabla_sens.loc[tabla_sens["eps (km)"] == 3, "% ruido"].mean() > tabla_sens.loc[tabla_sens["eps (km)"] == 20, "% ruido"].mean(), "Más eps debería dejar menos ruido."
print("✓ E3.2 completo")

## Lista de cotejo (autoevaluación)

- [ ] Tu DBSCAN coincide con scikit-learn (núcleos y ruido).
- [ ] Elegiste eps con el gráfico de k-distancia.
- [ ] Comparaste DBSCAN con K-means en datos no convexos.
- [ ] Agrupaste coordenadas con haversine y eps en kilómetros.
- [ ] Mediste la sensibilidad a eps y minPts y la estabilidad.

**Reflexión:** ¿qué pregunta de negocio fija el valor de eps en un problema geográfico de tu organización?

**Cómo te prepara para la Tarea 3:** la tarea permite segmentar con K-means o DBSCAN y exige justificar parámetros, validar y medir estabilidad.

## Desafío opcional con IA agéntica · HDBSCAN: zonas con densidades distintas

**Objetivo.** Probar HDBSCAN en el caso de California y compararlo con DBSCAN.

**Prompt inicial.** Úsalo en la herramienta que prefieras (Claude Code, Codex, Gemini en Colab, ChatGPT…), con este notebook resuelto como contexto. Pide primero un plan y revisa cada paso antes de aprobarlo.

```text
En el notebook resuelto E3.2 (DBSCAN con las coordenadas de California y haversine), agrega una sección con sklearn.cluster.HDBSCAN sobre las mismas coordenadas en radianes y metric="haversine". Compara con DBSCAN: número de zonas, porcentaje de puntos marcados como ruido y un mapa de cada solución. Explica en español qué problema de DBSCAN resuelve HDBSCAN y qué parámetros quedan por decidir.
```

**Cómo verificar el resultado**

- Las coordenadas están en radianes para ambos métodos.
- Se reportan las zonas y el porcentaje de ruido de cada solución.
- La explicación relaciona el resultado con las densidades distintas.

**Declara el uso de IA** (norma f del sílabo): herramienta, prompts relevantes, qué verificaste tú y qué corregiste. El desafío es opcional y no se califica; lo que cuenta es que puedas explicar cada decisión.